### Cross-phylum trait prediction from social-niche embeddings.
 
For each embedding (5 repeats x 6 training-set sizes) a random forest is
trained on Traitor labels and evaluated on BacDive labels, holding out one
focal phylum at a time.

In [70]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [74]:
# --------------------------------------------------------------------------- #
# configuration
# --------------------------------------------------------------------------- #
REF_EMB_TMPL = "data/embedding/subset_table_8w_100_{n}.txt"   # defines the ID universe
SUBSET_TMPL  = "data/embedding/subset_table_{t}_100_{n}.txt"
FULL_EMB_DIR = "../../data"     
TAXMAP       = "../../data/taxmap_slv_ssu_ref_nr_138.2.txt"
TRAITOR_CSV  = "data/trait_predcit.csv"
BACDIVE_CSV  = "data/bacDive.csv"
OUT_CSV      = "data/auc_res.csv"
 
REPEATS = [1, 2, 3, 4, 5]
DATASIZES = {"1w": "10,000", "2w": "20,000", "4w": "40,000",
             "8w": "80,000", "16w": "160,000", "21w": "210,000"}

In [75]:
# Embeddings covering the full ID set. Only these are evaluated at 21w; the
# smaller data sizes exist for social_niche only.
FULL_EMBS = {
    "social_niche": "social_niche_embedding_100.txt",
    "dnabert2":     "dnabert2_16s_embedding_reduced_100.txt",
    "phylo_pca":    "phylo_embed_PCA_100.txt",
}
 
BIG4 = ["Bacillota", "Bacteroidota", "Actinomycetota", "Pseudomonadota"]
BIG3 = BIG4[:3]

In [76]:
# trait -> (focal phyla, min samples per test class, min classes in test set)
TRAITS = {
    "Oxygen_Preference": (BIG4, 4, 3),
    "Gram_Status":       (BIG3, 6, 2),
    "Motility":          (BIG3, 6, 2),
    "Spore_Formation":   (BIG4, 6, 2),
}
 
MODEL = RandomForestClassifier(n_estimators=1000, random_state=0, oob_score=True,
                               n_jobs=-1, class_weight="balanced")

In [77]:
# --------------------------------------------------------------------------- #
# loading helpers
# --------------------------------------------------------------------------- #
def load_embedding(path):
    emb = pd.read_csv(path, header=None, sep=" ", low_memory=False, index_col=0)
    return emb
 
 
_full_cache = {}
 
 
def get_full_embedding(name):
    """Full-size embeddings do not depend on the repeat, so load each one once."""
    if name not in _full_cache:
        _full_cache[name] = load_embedding(f"{FULL_EMB_DIR}/{FULL_EMBS[name]}")
    return _full_cache[name]
 
 
def iter_embeddings(n):
    """Yield (embedding name, data-size key, embedding) for one repeat."""
    for t in DATASIZES:
        if t == "21w":
            for name in FULL_EMBS:
                yield name, t, get_full_embedding(name)
        else:
            yield "social_niche", t, load_embedding(SUBSET_TMPL.format(t=t, n=n))
 
 
def load_taxonomy():
    tax = pd.read_csv(TAXMAP, sep="\t", low_memory=False)
    # accession = "<col0>.<col1>.<col2>" (vectorised instead of a per-row loop)
    acc = (tax.iloc[:, 0].astype(str) + "." +
           tax.iloc[:, 1].astype(str) + "." +
           tax.iloc[:, 2].astype(str))
    ranks = tax["path"].str.split(";", expand=True).iloc[:, :7]
    ranks.columns = ["k", "p", "c", "o", "f", "g", "s"]
    ranks.index = acc.values
    return ranks
 
 
def combine_labels(df, mapping, exclusive=False):
    """Collapse indicator columns into one categorical column.
 
    mapping: {column: label}, applied in order (first match wins).
    exclusive=True -> NaN unless exactly one indicator is set.
    """
    out = pd.Series(np.nan, index=df.index, dtype=object)
    for col, label in mapping.items():
        out[out.isna() & (df[col] == 1)] = label
    if exclusive:
        out[df[list(mapping)].sum(axis=1) != 1] = np.nan
    return out
 
 
def load_traitor():
    tr = pd.read_csv(TRAITOR_CSV, index_col=0).astype(int).replace(3, 1)
    return pd.DataFrame({
        "Oxygen_Preference": combine_labels(
            tr, {"Aerobe": "aerobic", "Facultative": "facultatively",
                 "Anaerobe": "anaerobic"}, exclusive=True),
        "Gram_Status": combine_labels(
            tr, {"Gram negative": "negative", "Gram positive": "positive"},
            exclusive=True),
        "Motility": tr["Motile"],
        "Spore_Formation": tr["Spore formation"],
    })
 
 
def load_bacdive():
    tr = (pd.read_csv(BACDIVE_CSV)
            .drop_duplicates(subset="16s_ID")
            .set_index("16s_ID")
            .replace({"NA": np.nan, "": np.nan, "-": "no", "+": "yes",
                      "+;NA": np.nan, "mixed": np.nan, "variable": np.nan,
                      "no;yes": np.nan, "negative;positive": np.nan,
                      "negative;variable": np.nan}))
    out = pd.DataFrame({
        "Oxygen_Preference": combine_labels(
            tr, {"aerobe": "aerobic", "facultative.anaerobe": "facultatively",
                 "anaerobe": "anaerobic"}),
        "Gram_Status": tr["gram_stain"],
        "Motility": tr["motility"],
        "Spore_Formation": tr["spore_formation"],
    })
    return out.replace({"yes": 1, "no": 0})
 
 
def align_bacdive(bacdive, fid):
    """Re-index BacDive (keyed by bare accession) onto embedding IDs."""
    acc = np.array([str(x).split(".")[0] for x in fid])
    keep = np.isin(acc, bacdive.index.values)
    out = bacdive.loc[acc[keep]]
    out.index = fid[keep]
    return out

In [78]:
# --------------------------------------------------------------------------- #
# evaluation
# --------------------------------------------------------------------------- #
def score_split(emb, tax, traitor, bacdive, trait, phylum, focal, min_count, min_classes):
    """Train on Traitor, test on BacDive, holding out `phylum` + all non-focal phyla."""
    test_lab = bacdive[trait].dropna()
    tax_test = tax.loc[test_lab.index]
    held_out = set(p for p in tax_test["p"].unique() if p not in focal) | {phylum}
 
    test_id = tax_test.index[tax_test["p"].isin(held_out)]
    train_lab = traitor[trait].dropna()
    tax_train = tax.loc[train_lab.index]
    train_id = tax_train.index[~tax_train["p"].isin(held_out)]
 
    y_test = test_lab.loc[test_id].values
    classes, counts = np.unique(y_test, return_counts=True)
    if len(classes) < min_classes or counts.min() < min_count:
        return None
 
    MODEL.fit(emb.loc[train_id], train_lab.loc[train_id].values)
    proba = MODEL.predict_proba(emb.loc[test_id])
    if not set(classes).issubset(MODEL.classes_):
        return None                                   # test label unseen in training
 
    if len(MODEL.classes_) > 2:
        return roc_auc_score(y_test, proba, multi_class="ovr", average="macro",
                             labels=MODEL.classes_)
    positive = MODEL.classes_[1]
    return roc_auc_score((y_test == positive).astype(int), proba[:, 1])

### run model predict Gram_Status, Oxygen_Preference, Cell_Shape, Spore_Formation, Motility

In [79]:
fid_ref = np.unique(np.concatenate(
    [load_embedding(REF_EMB_TMPL.format(n=n)).index.values for n in REPEATS]))
 
tax_all = load_taxonomy()
traitor_all = load_traitor()
bacdive_all = load_bacdive()

records = []
for n in REPEATS:
    for emb_name, t, emb in iter_embeddings(n):
        
        fid_ref = fid_ref[fid_ref != '<unk>']
        fid = np.intersect1d(fid_ref, emb.index.values)
        emb = emb.loc[fid]

        tax = tax_all.loc[fid]
        traitor = traitor_all.loc[np.intersect1d(fid, traitor_all.index.values)]
        bacdive = align_bacdive(bacdive_all, fid)

        for trait, (focal, min_count, min_classes) in TRAITS.items():
            for phylum in focal:
                auc = score_split(emb, tax, traitor, bacdive, trait, phylum,
                                  focal, min_count, min_classes)
                if auc is not None:
                    records.append({"auc": auc, "embedding": emb_name,
                                    "traits_type": trait,
                                    "group": f"times_{n}", "tax": phylum,
                                    "datasize": DATASIZES[t]})

pd.DataFrame(records).to_csv(OUT_CSV, index=False)

/home/dongbiao/tmp/ipykernel_197581/2282388548.py:70: DtypeWarning: Columns (8,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,160,161,162,163,165,166,167,168,169,170,171,172,177,179,180,183,184,185,186,187,189,190,191,192,193,194,195,304,309,313,348,351,352,353,354,356,361,363,364,366,367,370,374,375,382,389,390,398,400,401,403,404,405,408,417,420,421,424,482,483,496,497,498,499,500,501,502,503,505,506,507,510,512,514,521,522,523,526,530,531,532,533,534,535,536,537,539,540,541,542,544,545,546,547,548,549,550,552,555,556,559,561,562,563,564,568,569,570,571,572,573,576,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673

#### predict metabolic

In [80]:
embed = pd.read_csv("../../data/social_niche_embedding_100.txt",
                          header=None, sep=" ", low_memory=False, index_col=0)
embed.drop("<unk>", inplace=True)
fid = embed.index.values

In [81]:
taxonomy = pd.read_csv("../../data/taxmap_slv_ssu_ref_nr_138.2.txt", sep="\t", low_memory=False)

acc = []
for i in range(taxonomy.shape[0]):
    temp = taxonomy.iloc[i]
    acc.append(f"{temp[0]}.{temp[1]}.{temp[2]}")

taxonomy = taxonomy.loc[:, "path"].str.split(';', expand=True)
taxonomy.index = acc
taxonomy = taxonomy.iloc[:, 0: 7]
taxonomy.columns = ["k", "p", "c", "o", "f", "g", "s"]
taxonomy = taxonomy.loc[fid]

In [102]:
metabolic = ["Lactose", "Melibiose", "Glycerol", "Maltose", "Trehalose", "Salicin", "Sucrose", "Sorbitol", "D-Sorbitol"]

In [106]:
traits =  pd.read_csv("data/trait_predcit.csv", index_col=0)
inter_id = np.intersect1d(fid, traits.index.values)
traits = traits.loc[inter_id]
traits = traits.astype(int)
traits[traits.values == 3] = 1
traits_traitor = traits.loc[:, [i in metabolic for i in traits.columns.values]]
traits_traitor.columns.values

array(['Lactose', 'Salicin', 'Glycerol', 'Melibiose', 'Maltose',
       'Sucrose', 'Trehalose', 'D-Sorbitol'], dtype=object)

In [139]:
traits_traitor.columns = ['Lactose', 'Salicin', 'Glycerol', 'Melibiose', 'Maltose', 'Sucrose', 'Trehalose', 'Sorbitol']

In [113]:
# --- 1. Load and prepare the traits data ---
# Read the CSV, drop duplicates based on 'X16s_ID', and set it as the index
traits = pd.read_csv("data/bacDive.csv", low_memory=False)
traits.drop_duplicates(subset='16s_ID', inplace=True)
traits.set_index('16s_ID', inplace=True)

# Extract the accession number by splitting the index string at the period '.'
accessions_num = co_embedding.index.str.split('.').str[0]

# Create a helper DataFrame to map the full embed_id to the shortened accession number
df_map = pd.DataFrame({
    'accessions': accessions_num,
    'embed_id': co_embedding.index
})

# --- 3. Align traits and embedding data ---
# Find the common accession numbers between the two datasets
inter_id = np.intersect1d(traits.index, df_map['accessions'])

# Filter the mapping and traits DataFrames to keep only the common entries
df_map = df_map[df_map['accessions'].isin(inter_id)]
traits = traits.loc[df_map['accessions']]

# Update the traits index to match the full embedding ID for consistency
traits.index = df_map['embed_id']

# --- 4. Clean and standardize data in the traits DataFrame ---
# Create a dictionary for all values that need to be replaced
replace_dict = {"NA": np.nan, "-": "no", "+": "yes",
    "": np.nan, "+;NA": np.nan, "coccus-shaped": "coccus",
    "rod-shaped": "rod", "mixed": np.nan, "negative;variable": np.nan,
    "no;yes": np.nan, "negative;positive": np.nan, "variable": np.nan
                
}
traits.replace(replace_dict, inplace=True)

# Standardize the 'cell_shape' column: keep only 'coccus' or 'rod', set others to NaN
valid_shapes = ['coccus', 'rod']
traits['cell_shape'] = traits['cell_shape'].where(traits['cell_shape'].isin(valid_shapes), np.nan)

# --- 5. Create the 'Oxygen.Preference' column ---
# Define conditions and corresponding choices for oxygen preference
conditions = [
    traits['aerobe'] == 1,
    traits['facultative.anaerobe'] == 1,
    traits['anaerobe'] == 1
]
choices = ['aerobic', 'facultatively', 'anaerobic']

# Use np.select (similar to R's case_when) to create the new column
# The default value is NaN for anything that doesn't meet a condition
traits['Oxygen.Preference'] = np.select(conditions, choices, default=np.nan)

# --- 6. Final cleanup ---
# Define columns to remove
remove_cols = ['X16s_ID', 'aerobe', 'facultative.anaerobe', 'anaerobe']
# Drop the specified columns; 'errors='ignore'' prevents an error if a column is already gone
traits.drop(columns=remove_cols, inplace=True, errors='ignore')

# Drop any column where all values are missing (NaN)
traits.dropna(axis=1, how='all', inplace=True)

In [116]:
agg_bac = pd.read_csv("data/agg_bac.csv")
agg_bac.level_3 = [i.capitalize() for i in agg_bac.level_3.values]
agg_bac = agg_bac.loc[[i in metabolic for i in agg_bac.level_3.values]]
agg_bac = agg_bac.loc[[i in ["builds_acid_from"] for i in agg_bac.level_2.values]]
agg_bac = agg_bac.loc[agg_bac.type == 1]

In [118]:
traits_bacdive = traits[agg_bac.terms.values]
traits_bacdive.columns = agg_bac.level_3.values

In [122]:
traits_bacdive.columns.values

array(['Maltose', 'Sucrose', 'Lactose', 'Trehalose', 'Salicin',
       'Melibiose', 'Glycerol', 'Sorbitol'], dtype=object)

In [90]:
names = ['Lactose', 'Salicin', 'Glycerol', 'Melibiose', 'Maltose', 'Sucrose', 'Trehalose', 'Sorbitol']

In [91]:
labels_dict = {"yes":1, "no": 0}
phylum_id = ["Bacillota", "Bacteroidota", "Actinomycetota"]

In [144]:
auc = []
traits_type = []
tax = []
embedding = []
for e in ["social_niche", "dnabert2", "phylo_pca"]:
    for j in names:
        for i in phylum_id:
            embed = pd.read_csv(f"../../data/{FULL_EMBS[e]}", header=None, sep=" ", low_memory=False, index_col=0)
            temp = traits_bacdive.dropna(subset=[j])
            # temp = temp.loc[temp.loc[:, j].values == temp.loc[:, j].values]
            tax_bacdive = taxonomy.loc[temp.index.values]
            phylum = tax_bacdive.p.unique()
            test_phylum = list(phylum[[i not in phylum_id for i in phylum]]) + [i]
            test_id = tax_bacdive.loc[[i in test_phylum for i in tax_bacdive.p]].index.values
            test_id = test_id[traits_bacdive.loc[test_id, j].values == traits_bacdive.loc[test_id, j].values]
            
            tax_traitor = taxonomy.loc[traits_traitor.index.values]
            phylum = tax_traitor.p.unique()
            train_id = tax_traitor.loc[[i not in test_phylum for i in tax_traitor.p]].index.values
        
            train_id = train_id[traits_traitor.loc[train_id, j].values == traits_traitor.loc[train_id, j].values]
            X_train = embed.loc[train_id]
            y_train = traits_traitor.loc[train_id, j].values
            X_test = embed.loc[test_id]
            y_test = traits_bacdive.loc[test_id, j].values
            y_test = [labels_dict[i] for i in y_test]
            
            countsunique_elements, counts = np.unique(y_test, return_counts=True)
            
            if np.all(counts >= 4) and len(countsunique_elements) > 1:
                rf_model.fit(X_train, y_train)
                y_pred = rf_model.predict(X_test)
                y_pred_proba = rf_model.predict_proba(X_test)
                auc.append(roc_auc_score(y_test, y_pred_proba[:, 1]))
                traits_type.append(j)
                tax.append(i)
                embedding.append(e)

In [146]:
res = pd.DataFrame({"AUC":auc, "embedding":embedding, "traits":traits_type, "tax":tax})

In [147]:
res.to_csv("data/predict_metabolics_res.csv", index=None)